In [3]:
import numpy as np
import random
import pickle

# ----------------------------------------------------
# 1. Tic-Tac-Toe Environment
# ----------------------------------------------------
class TicTacToe:
    def __init__(self):
        self.board = np.zeros((3, 3), int)

    def reset(self):
        self.board[:] = 0
        return self.get_state()

    def get_state(self):
        return tuple(self.board.flatten())

    def available_actions(self):
        return [(r, c) for r in range(3) for c in range(3) if self.board[r, c] == 0]

    def check_winner(self):
        lines = [
            *self.board,                                 # rows
            *self.board.T,                               # columns
            self.board.diagonal(),                       # main diagonal
            np.fliplr(self.board).diagonal()             # anti diagonal
        ]
        for player in (1, -1):
            if any(np.all(line == player) for line in lines):
                return player
        return 0 if not self.available_actions() else None

    def step(self, action, player):
        if self.board[action] != 0:
            return self.get_state(), -10, True

        self.board[action] = player
        winner = self.check_winner()

        if winner is not None:
            reward = 1 if winner == 1 else -1 if winner == -1 else 0.5
            return self.get_state(), reward, True

        return self.get_state(), 0, False

    def print_board(self):
        symbols = {1: "X", -1: "O", 0: "."}
        for row in self.board:
            print(" | ".join(symbols[x] for x in row))
        print("-" * 9)


# ----------------------------------------------------
# 2. Q-Learning Agent
# ----------------------------------------------------
class QLearningAgent:
    def __init__(self, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.q = {}
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def q_value(self, state, action):
        return self.q.get((state, action), 0.0)

    def choose_action(self, state, actions, exploit=False):
        if not actions:
            return None
        if not exploit and random.random() < self.epsilon:
            return random.choice(actions)

        q_vals = [self.q_value(state, a) for a in actions]
        max_q = max(q_vals)
        best = [a for a, q in zip(actions, q_vals) if q == max_q]
        return random.choice(best)

    def update(self, state, action, reward, next_state, next_actions):
        old = self.q_value(state, action)
        next_max = max((self.q_value(next_state, a) for a in next_actions), default=0.0)
        self.q[(state, action)] = old + self.alpha * (reward + self.gamma * next_max - old)

    def save(self):
        pickle.dump(self.q, open("q_table.pkl", "wb"))
        print("Q-table saved.")

    def load(self):
        try:
            self.q = pickle.load(open("q_table.pkl", "rb"))
            print("Q-table loaded.")
        except:
            print("No saved Q-table found.")


# ----------------------------------------------------
# 3. Training
# ----------------------------------------------------
def train(agent, episodes=30000):
    env = TicTacToe()

    for ep in range(episodes):
        state = env.reset()
        done = False

        while not done:
            # Agent move (X = 1)
            actions = env.available_actions()
            action = agent.choose_action(state, actions)
            next_state, reward, done = env.step(action, 1)

            if done:
                agent.update(state, action, reward, next_state, [])
                break

            # Opponent move (random O = -1)
            opp_action = random.choice(env.available_actions())
            next2, reward2, done2 = env.step(opp_action, -1)

            if done2:
                agent.update(state, action, reward2, next2, [])
                break

            agent.update(state, action, 0, next2, env.available_actions())
            state = next2

        if (ep + 1) % 5000 == 0:
            print("Training:", ep + 1)

    agent.save()


# ----------------------------------------------------
# 4. Play Against the Trained Agent
# ----------------------------------------------------
def play(agent):
    env = TicTacToe()
    state = env.reset()

    human = input("Play as X or O? ").upper()
    human_p = 1 if human == "X" else -1
    agent_p = -human_p
    turn = 1  # X starts

    while True:
        env.print_board()

        if turn == agent_p:
            print("Agent's move...")
            action = agent.choose_action(state, env.available_actions(), exploit=True)
            state, reward, done = env.step(action, agent_p)

        else:
            while True:
                try:
                    r, c = map(int, input("Your move (r,c): ").split(','))
                    if (r, c) in env.available_actions():
                        break
                    print("Invalid move!")
                except:
                    print("Enter like: 0,2")
            state, reward, done = env.step((r, c), human_p)

        if done:
            env.print_board()
            if reward == 0.5:
                print("Draw!")
            elif (reward == 1 and agent_p == 1) or (reward == -1 and agent_p == -1):
                print("Agent Wins!")
            else:
                print("You Win!")
            return

        turn *= -1


# ----------------------------------------------------
# 5. Run
# ----------------------------------------------------
if __name__ == "__main__":
    agent = QLearningAgent()
    train(agent, 30000)
    play(agent)


Training: 5000
Training: 10000
Training: 15000
Training: 20000
Training: 25000
Training: 30000
Q-table saved.


Play as X or O?  X


. | . | .
. | . | .
. | . | .
---------


Your move (r,c):  1,1


. | . | .
. | X | .
. | . | .
---------
Agent's move...
. | O | .
. | X | .
. | . | .
---------


Your move (r,c):  2,2


. | O | .
. | X | .
. | . | X
---------
Agent's move...
. | O | .
. | X | .
. | O | X
---------


Your move (r,c):  0,0


X | O | .
. | X | .
. | O | X
---------
You Win!
